### Run this notebook online

[![Open in Colab](https://img.shields.io/badge/Open_in-Colab-F9AB00?logo=googlecolab&logoColor=F9AB00)](https://colab.research.google.com/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/03_CIFAR10_extra_joint.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2Fhosein-fanai%2FContinual-Learning-with-Diffusion-Vision-Transformers%2Fblob%2Fmain%2Fnotebooks%2Fthesis%2F03_CIFAR10_extra_joint.ipynb)
[![Launch Binder](https://img.shields.io/badge/launch-binder-F5793A?logo=jupyter&logoColor=white)](https://mybinder.org/v2/gh/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/main?urlpath=lab%2Ftree%2Fnotebooks%2Fthesis%2F03_CIFAR10_extra_joint.ipynb)

- **Google Colab:** open the notebook, select a GPU for training under **Runtime > Change runtime type**, then choose **Run all**.
- **Kaggle:** sign in and import the notebook, enable **Internet**, select a **GPU** accelerator for training, then **Run all**.
- **Binder:** opens a temporary CPU JupyterLab session. Use it to inspect the notebook or run small checks; full training needs more resources.
- **[Studio Lab](https://studiolab.sagemaker.aws/import/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/03_CIFAR10_extra_joint.ipynb) (existing accounts only):** start a runtime, copy the notebook to your project, select a Python **3.11–3.13** kernel, and set `RUNTIME = "studiolab"` in the first code cell before **Run all**. For CPU, also set `CUDA = False`.

The **first code cell** finds or downloads the repository and prepares TensorFlow **2.20** / Keras **3.11.2** before project imports. If setup requests a restart, restart the kernel and run all again. For another hosted Jupyter service, set `RUNTIME = "hosted"` (`CUDA = False` for CPU or compatible provider-managed CUDA). Locally, select the project TensorFlow kernel.

Launch links open the published GitHub `main` version; publish this notebook and its setup files together before using them. For a notebook that has not been published, upload its `.ipynb` file to Colab or Kaggle instead. GPU availability depends on the provider. Save checkpoints and results before a temporary session ends.

Training notebooks 03–09 require the prepared 21-stream campaign artifacts. Notebook **01** reproduces that preparation; skip it when the campaign is already supplied. Notebook **10** is optional collection.

See the [hosted runtime guide](https://github.com/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/README.md#hosted-runtimes) for setup and import details.


In [ ]:
# Shared setup: use the local initializer when available, otherwise download it.
from pathlib import Path
from urllib.request import urlopen


CHECKOUT_NAME = "Continual-Learning-with-Diffusion-Vision-Transformers"
REPOSITORY = f"https://github.com/hosein-fanai/{CHECKOUT_NAME}.git"
REVISION = "main"
RUNTIME = "auto"  # Use "hosted" for another online service, or "local" to verify only.
CUDA = None  # False: CPU or managed CUDA; True: retain CUDA pip dependencies.

_locations = (Path.cwd(), *Path.cwd().parents, Path.cwd() / CHECKOUT_NAME,
              Path("/kaggle/working") / CHECKOUT_NAME, Path("/content") / CHECKOUT_NAME)
_initializer = next((path / "notebooks" / "init.py" for path in _locations
                     if (path / "notebooks" / "init.py").is_file()), None)
_url = f"https://raw.githubusercontent.com/hosein-fanai/{CHECKOUT_NAME}/{REVISION}/notebooks/init.py"
_setup = {"__name__": "notebook_setup", "__file__": str(_initializer or _url)}
with (_initializer.open("rb") if _initializer else urlopen(_url, timeout=30)) as _file:
    exec(compile(_file.read(), _setup["__file__"], "exec"), _setup)
ROOT, RUNTIME_PACKAGES = _setup["prepare_notebook"](
    checkout_name=CHECKOUT_NAME, repository=REPOSITORY, revision=REVISION,
    runtime=RUNTIME, cuda=CUDA,
)


# 03 — CIFAR10 / Extra joint

Extra ordinary training with the same additional update allowance as the semantic phases.

Select a **TensorFlow 2.20 / Keras 3** kernel, restart the kernel, then **Run All**.

In [ ]:
from notebooks.thesis.workflow import check_runtime


print(check_runtime())
from IPython import get_ipython


get_ipython().run_line_magic("matplotlib", "inline")
from common.dataloader import get_datasets
from common.model import get_model
from common.train import train_model
from notebooks.thesis.workflow import load_run, attach_route, close_run, finish_run
from notebooks.thesis.presentation import describe_run, show_learning_results, show_diagnostics, show_saved_replay


CAMPAIGN = ROOT / "results/thesis_route_one/minimum_v6_tf220_21streams"

### 1. Select one stream

Use the prepared `minimum_v6_tf220_21streams` campaign with the approved recipe. Notebook 01 need not be rerun when those artifacts already exist. Each launch selects one next unfinished paired repeat automatically. Complete three fresh-kernel launches of this notebook across the saved checklist.

In [ ]:
SHOW_DIAGNOSTICS = False  # Optional saved validation/replay views; never change the frozen recipe.
config, context = load_run(CAMPAIGN / "frozen_design.json", "cifar10", "extra_joint",
                           repeat_index=None)
describe_run(config)

### 2. Load data and create the model

The common APIs own splitting, replay and class growth. The route attaches to this same model.

In [ ]:
project = config.common
trainset, valset = get_datasets(project)
bundle = get_model(project)
attach_route(context, bundle)

### 3. Train once

After an interruption, restart the kernel and **Run All** to resume the same stream. Work after the latest valid checkpoint is repeated.

In [ ]:
if context.get("training_started"):
    raise RuntimeError("Restart the kernel before training another stream.")
context["training_started"] = True
try:
    history = train_model(project, bundle, trainset, valset=valset)
except BaseException:
    close_run(context, release=True)
    raise
finally:
    close_run(context)

### 4. Save and read the results

Accuracy is percent; forgetting and backward transfer are signed percentage points. These are test outcomes from one complete stream.

In [ ]:
evaluations = finish_run(context, config, bundle, history, trainset, valset)
RUN, VIEW_DIR = show_learning_results(config, bundle, details=SHOW_DIAGNOSTICS)

### Optional: saved diagnostics

Enabled by SHOW_DIAGNOSTICS above. Phase changes use matched validation examples; replay grids are qualitative. These views are also available from notebook 10.

In [ ]:
if SHOW_DIAGNOSTICS:
    review = show_diagnostics(RUN, VIEW_DIR)
    show_saved_replay(config, bundle, RUN, VIEW_DIR)

Follow the next row in the saved execution checklist using a fresh kernel. Each notebook 03–09 selects one next unfinished repeat per launch: three completed launches each give **21 streams**. Notebook 02 and supplemental notebooks 11/12 are outside this plan. After all 21 streams finish, notebook 10 can optionally collect the saved results. Keep negative or uncertain results and do not change the frozen design.